# 05_Modeling (v2)

Notebook ini menyiapkan pipeline modeling untuk LSTM univariat (contoh untuk satu komoditas). Termasuk: persiapan dataset, dataset class, model (menggunakan `src/models/base_model.py`), loop training singkat untuk validasi end-to-end, dan evaluasi metrik dasar.

## Konfigurasi dan Persiapan

Notebook menggunakan parameter dari `config.yaml`.

In [11]:
import yaml
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler

# Load config
cfg = yaml.safe_load(Path('../config.yaml').read_text())
model_cfg = cfg['model']['univariate']
seq_len = model_cfg.get('sequence_length', 28)
hidden_size = model_cfg.get('hidden_size', 64)
num_layers = model_cfg.get('num_layers', 2)
dropout = model_cfg.get('dropout', 0.2)
lr = model_cfg.get('learning_rate', 1e-3)
batch_size = model_cfg.get('batch_size', 64)
epochs = 60 
use_cuda = cfg.get('training', {}).get('use_cuda', True) and torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
print('Device:', device)

Device: cuda


In [12]:
# Jalankan training langsung dari notebook menggunakan skrip `src/train.py`
import subprocess, sys
import yaml
from pathlib import Path
project_root = Path('..').resolve()
compare_csv = project_root / 'runs' / 'all_commodity_compare.csv'
compare_csv.parent.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load((project_root / 'config.yaml').read_text())
batch_size = cfg['model']['univariate'].get('batch_size', 64)
epochs = 60

# Set one commodity to run here; set run_all = True to execute every commodity in config
train_commodity = 'Beras Medium'
run_all = True
commodities = [c['name'] for c in cfg['commodities']] if run_all else [train_commodity]

for series in commodities:
    print('\n=== Running commodity:', series, '===')
    cmd = [
        sys.executable,
        str(project_root / 'src' / 'train.py'),
        '--commodity', series,
        '--mode', 'both',
        '--epochs', str(epochs),
        '--batch-size', str(batch_size),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
        '--log-dir', str(project_root / f'runs/{series.replace(" ", "_")}'),
        '--log-csv', str(project_root / 'runs' / f'metrics_{series.replace(" ", "_")}.csv'),
        '--compare-csv', str(compare_csv),
        '--early-stopping-patience', '10',
    ]
    try:
        subprocess.run(cmd, cwd=str(project_root), check=True)
    except subprocess.CalledProcessError as e:
        print('Error running', series, e)
        continue

print('\nAll runs finished. Loading comparison CSV:')
import pandas as _pd
if compare_csv.exists():
    comp_df = _pd.read_csv(compare_csv)
    display(comp_df)
    comp_df
else:
    print('Comparison CSV not found at', compare_csv)


=== Running commodity: Beras Premium ===


2026-06-02 15:17:40.127912: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 15:17:40.168730: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-02 15:17:40.186477: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-02 15:17:40.191114: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-02 15:17:40.212958: I tensorflow/core/platform/cpu_feature_guar

Device: cuda
Epoch 1/60 - train_loss=0.794795 val_loss=0.687540
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/univariate/lstm_Beras_Premium_univariate.pth
Epoch 2/60 - train_loss=0.180737 val_loss=0.151879
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/univariate/lstm_Beras_Premium_univariate.pth
Epoch 3/60 - train_loss=0.081327 val_loss=0.042371
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/univariate/lstm_Beras_Premium_univariate.pth
Epoch 4/60 - train_loss=0.054828 val_loss=0.037200
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/univariate/lstm_Beras_Premium_univariate.pth
Epoch 5/60 - train_loss=0.042210 val_loss=0.072270
Epoch 6/60 - train_loss=0.035821 val_loss=0.034931
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 1/60 - train_loss=0.213375 val_loss=0.172638
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/multivariate/lstm_Beras_Premium_multivariate.pth
Epoch 2/60 - train_loss=0.018001 val_loss=0.169756
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/multivariate/lstm_Beras_Premium_multivariate.pth
Epoch 3/60 - train_loss=0.006877 val_loss=0.151288
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/multivariate/lstm_Beras_Premium_multivariate.pth
Epoch 4/60 - train_loss=0.005305 val_loss=0.154378
Epoch 5/60 - train_loss=0.005131 val_loss=0.154107
Epoch 6/60 - train_loss=0.004703 val_loss=0.150605
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Beras_Premium/multivariate/lstm_Beras_Premium_multivariate.pth
Epoch 7/60 - train_loss=0.004461 val_loss=0.142836
  Saved best model to /home/rna_13/Data

...

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 1/60 - train_loss=0.584981 val_loss=0.027286
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Kacang_Hijau/multivariate/lstm_Kacang_Hijau_multivariate.pth
Epoch 2/60 - train_loss=0.333676 val_loss=0.000536
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Kacang_Hijau/multivariate/lstm_Kacang_Hijau_multivariate.pth
Epoch 3/60 - train_loss=0.238313 val_loss=0.000135
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/Kacang_Hijau/multivariate/lstm_Kacang_Hijau_multivariate.pth
Epoch 4/60 - train_loss=0.161439 val_loss=0.001878
Epoch 5/60 - train_loss=0.133258 val_loss=0.001256
Epoch 6/60 - train_loss=0.123578 val_loss=0.002219
Epoch 7/60 - train_loss=0.103688 val_loss=0.000159
Epoch 8/60 - train_loss=0.090899 val_loss=0.000385
Epoch 9/60 - train_loss=0.089259 val_loss=0.000597
Epoch 10/60 - train_loss=0.074464 val_loss=0.001146
Epoch 11/60 - train_loss=0.076975 val_loss

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path


All runs finished. Loading comparison CSV:


,commodity,mode,epochs,mae,rmse,mape,smape,directional_acc
0,Beras Premium,univariate,60,319.854,320.032,2.460,2.430,0.0
1,Beras Premium,multivariate,60,494.515,494.543,3.804,3.733,0.0
2,Beras Premium,univariate,60,244.661,244.789,1.882,1.864,0.0
3,Beras Premium,multivariate,60,745.200,745.202,5.732,5.573,0.0
4,Beras Medium,univariate,60,232.873,233.183,1.863,1.846,0.0
5,Beras Medium,multivariate,60,11.611,13.076,0.093,0.093,0.0
6,Jagung Pipil Kering,univariate,60,45.298,48.863,1.161,1.169,0.0
7,Jagung Pipil Kering,multivariate,60,161.012,161.804,4.129,4.044,0.0
8,Kacang Hijau,univariate,60,8.497,10.178,0.065,0.065,0.0
9,Kacang Hijau,multivariate,60,5.028,6.067,0.039,0.039,0.0


## Evaluasi Hasil Training
Setelah menjalankan training dari notebook, muat file perbandingan yang tersedia untuk menilai performa dan memilih komoditas yang perlu tuning.

In [21]:
from pathlib import Path
import pandas as pd

project_root = Path('..').resolve()
compare_path = project_root / 'runs' / 'recomputed_compare.csv'
if not compare_path.exists():
    compare_path = project_root / 'runs' / 'all_commodity_compare.csv'

print('Loading comparison file:', compare_path)
if compare_path.exists():
    comp_df = pd.read_csv(compare_path)
    display(comp_df)
    if {'commodity','mode','mae','rmse','smape'}.issubset(comp_df.columns):
        summary = comp_df.pivot(index='commodity', columns='mode', values='mae').reset_index()
        summary.columns.name = None
        display(summary)
else:
    print('No comparison CSV found. Please run the training cell first.')

Loading comparison file: /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/recomputed_compare.csv


,commodity,mode,epochs,mae,rmse,mape,smape,directional_acc
0,Beras Premium,univariate,60,183.995,184.167,1.415,1.405,0.0
1,Beras Premium,multivariate,60,465.147,465.448,3.578,3.515,0.0
2,Beras Medium,univariate,60,70.425,71.493,0.563,0.562,0.0
3,Beras Medium,multivariate,60,44.176,45.179,0.353,0.353,0.0
4,Jagung Pipil Kering,univariate,60,91.250,93.407,2.340,2.369,0.0
5,Jagung Pipil Kering,multivariate,60,117.196,118.027,3.005,2.960,0.0
6,Kacang Hijau,univariate,60,9.844,10.925,0.076,0.076,0.0
7,Kacang Hijau,multivariate,60,3.378,4.004,0.026,0.026,0.0


,commodity,multivariate,univariate
0,Beras Medium,44.176,70.425
1,Beras Premium,465.147,183.995
2,Jagung Pipil Kering,117.196,91.250
3,Kacang Hijau,3.378,9.844


## Ringkasan Perbandingan Hasil
Selanjutnya, kita tinjau mode terbaik per komoditas menurut MAE dan rata-rata performa global untuk setiap mode.

In [14]:
if compare_path.exists():
    best_by_mae = (
        comp_df.sort_values(['commodity', 'mae'])
        .groupby('commodity', as_index=False)
        .first()
        .loc[:, ['commodity', 'mode', 'mae', 'rmse', 'smape']]
    )
    print('Best mode per commodity menurut MAE:')
    display(best_by_mae)

    agg = comp_df.groupby('mode').agg(
        avg_mae=('mae', 'mean'),
        avg_rmse=('rmse', 'mean'),
        avg_smape=('smape', 'mean'),
    ).reset_index()
    print('Rata-rata performa per mode:')
    display(agg)
else:
    print('No comparison CSV found. Please run the training cell first.')

Best mode per commodity menurut MAE:


,commodity,mode,mae,rmse,smape
0,Beras Medium,multivariate,44.176,45.179,0.353
1,Beras Premium,univariate,183.995,184.167,1.405
2,Jagung Pipil Kering,univariate,91.250,93.407,2.369
3,Kacang Hijau,multivariate,3.378,4.004,0.026


Rata-rata performa per mode:


,mode,avg_mae,avg_rmse,avg_smape
0,multivariate,157.47425,158.1645,1.7135
1,univariate,88.87850,89.9980,1.1030


## Load Data Terproses

Menggunakan `data/processed/price_cleaned.csv` yang dibuat di tahap Data Preparation.

In [15]:
processed_dir = Path('../data/processed')
price_df = pd.read_csv(processed_dir / 'price_cleaned.csv', parse_dates=['Tanggal'])
price_df = price_df.sort_values(['Komoditi', 'Tanggal']).reset_index(drop=True)
price_df.head(3)

,Komoditi,Tanggal,Harga Petani,Harga Pengecer
0,Beras Medium,2022-01-01,8800.0,9000.0
1,Beras Medium,2022-01-02,8800.0,9000.0
2,Beras Medium,2022-01-03,8800.0,9000.0


## Dataset Sequence (Univariat)

Membangun dataset sliding-window untuk target `Harga Petani`. Contoh menggunakan satu komoditas: `Beras Medium`.

In [16]:
class PriceSequenceDataset(Dataset):
    def __init__(self, features: np.ndarray, targets: np.ndarray, seq_len: int, scaler_x=None, scaler_y=None):
        self.seq_len = seq_len
        self.scaler_x = scaler_x or StandardScaler()
        self.scaler_y = scaler_y or StandardScaler()
        self.features = self.scaler_x.fit_transform(features.astype(float))
        self.targets = self.scaler_y.fit_transform(targets.reshape(-1, 1)).ravel()

    def __len__(self):
        return len(self.features) - self.seq_len

    def __getitem__(self, idx):
        x = self.features[idx: idx + self.seq_len]
        y = self.targets[idx + self.seq_len]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

    def inverse_target(self, y):
        arr = np.array(y).reshape(-1, 1)
        return self.scaler_y.inverse_transform(arr).ravel()

# Select one commodity series
series_name = cfg['commodities'][1]['series_id']  # Beras Medium (index 1)
series = price_df.loc[price_df['Komoditi'] == series_name].set_index('Tanggal')[cfg['preprocessing']['target_col']]
print('Series length for', series_name, len(series))

# Build univariate dataset and dataloaders
features = series.values.reshape(-1, 1).astype(float)
targets = series.values.astype(float)
full_dataset = PriceSequenceDataset(features, targets, seq_len)
print('Univariate feature dimension:', full_dataset.features.shape[1])

n = len(full_dataset)
train_n = int(n * 0.7)
val_n = int(n * 0.15)
test_n = n - train_n - val_n
train_ds, val_ds, test_ds = random_split(full_dataset, [train_n, val_n, test_n], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
print('Univariate dataset sizes (train/val/test):', len(train_ds), len(val_ds), len(test_ds))

Series length for Beras Medium 1064
Univariate feature dimension: 1
Univariate dataset sizes (train/val/test): 725 155 156


## Dataset Sequence (Multivariat dengan Fitur Produksi)

Bangun dataset sequence multivariat dengan menambahkan fitur produksi per kuartal sebagai eksogen. Sekarang model dapat melihat pola harga plus data produksi kuartal yang relevan.

In [20]:
prod_df = pd.read_csv(processed_dir / 'production_transformed.csv', parse_dates=['quarter_start'])
prod_df['quarter_start'] = pd.to_datetime(prod_df['quarter_start'])

series_df = price_df.loc[price_df['Komoditi'] == series_name].copy()
series_df['quarter_start'] = series_df['Tanggal'].dt.to_period('Q').dt.to_timestamp()

merged = series_df.merge(
    prod_df,
    left_on=['Komoditi', 'quarter_start'],
    right_on=['Komoditi_Harga', 'quarter_start'],
    how='left'
)

print('Merged shape:', merged.shape)
print('Sample merged columns:', merged.columns.tolist())

exog_cols = [
    c for c in merged.columns
    if c not in ['Tanggal', 'Komoditi', 'quarter_start', 'Komoditi_Harga', cfg['preprocessing']['target_col']]
    and np.issubdtype(merged[c].dtype, np.number)
]
input_cols = [cfg['preprocessing']['target_col']] + exog_cols
print('Input columns for multivariate model:', input_cols)

mv_features = merged[input_cols].ffill().bfill().values.astype(float)
mv_targets = merged[cfg['preprocessing']['target_col']].values.astype(float)
full_dataset_mv = PriceSequenceDataset(mv_features, mv_targets, seq_len)
print('Multivariate feature dimension:', full_dataset_mv.features.shape[1])

n_mv = len(full_dataset_mv)
train_n_mv = int(n_mv * 0.7)
val_n_mv = int(n_mv * 0.15)
test_n_mv = n_mv - train_n_mv - val_n_mv
train_ds_mv, val_ds_mv, test_ds_mv = random_split(full_dataset_mv, [train_n_mv, val_n_mv, test_n_mv], generator=torch.Generator().manual_seed(42))
train_loader_mv = DataLoader(train_ds_mv, batch_size=batch_size, shuffle=True)
val_loader_mv = DataLoader(val_ds_mv, batch_size=batch_size, shuffle=False)
test_loader_mv = DataLoader(test_ds_mv, batch_size=batch_size, shuffle=False)
print('Multivariate dataset sizes (train/val/test):', len(train_ds_mv), len(val_ds_mv), len(test_ds_mv))

Merged shape: (1064, 12)
Sample merged columns: ['Komoditi_x', 'Tanggal', 'Harga Petani', 'Harga Pengecer', 'quarter_start', 'Komoditi_y', 'tahun', 'triwulan', 'Luas Panen', 'Produktivitas', 'Produksi', 'Komoditi_Harga']
Input columns for multivariate model: ['Harga Petani', 'Harga Pengecer', 'tahun', 'triwulan', 'Luas Panen', 'Produktivitas', 'Produksi']
Multivariate feature dimension: 7
Multivariate dataset sizes (train/val/test): 725 155 156


## Perbandingan Metrik: Univariat vs Multivariat

Di bawah ini kita akan melatih dua model LSTM pendek dengan konfigurasi yang sama, lalu membandingkan metrik pada test set untuk model univariat dan multivariat. Ini membantu mengevaluasi apakah fitur produksi eksogen memberikan nilai tambah pada prediksi harga.

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import sys
from pathlib import Path as _Path
# ensure project root is on sys.path so `src` package is importable from notebooks
sys.path.insert(0, str(_Path('..').resolve()))
from src.models.base_model import BaseLSTMModel

def directional_accuracy(y_true, y_pred):
    dir_true = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    return 100.0 * np.mean(dir_true == dir_pred)

def train_model(model, loader, epochs, device, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device).unsqueeze(-1)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def evaluate_model(model, loader, scaler_y):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy().ravel()
            y_pred.extend(preds.tolist())
            y_true.extend(yb.numpy().ravel().tolist())
    y_true = scaler_y.inverse_transform(np.array(y_true).reshape(-1, 1)).ravel()
    y_pred = scaler_y.inverse_transform(np.array(y_pred).reshape(-1, 1)).ravel()
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "mape": np.mean(np.abs((y_pred - y_true) / np.where(y_true == 0, 1e-8, y_true))) * 100.0,
        "smape": np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + 1e-8)) * 100.0,
        "directional_accuracy": directional_accuracy(y_true, y_pred),
    }

# Create separate univariate loaders for comparison
n_uni = len(full_dataset)
train_n_uni = int(n_uni * 0.7)
val_n_uni = int(n_uni * 0.15)
test_n_uni = n_uni - train_n_uni - val_n_uni
train_ds_uni, val_ds_uni, test_ds_uni = random_split(full_dataset, [train_n_uni, val_n_uni, test_n_uni], generator=torch.Generator().manual_seed(42))
train_loader_uni = DataLoader(train_ds_uni, batch_size=batch_size, shuffle=True)
val_loader_uni = DataLoader(val_ds_uni, batch_size=batch_size, shuffle=False)
test_loader_uni = DataLoader(test_ds_uni, batch_size=batch_size, shuffle=False)

uni_model = BaseLSTMModel(input_size=full_dataset.features.shape[1], hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
mv_model = BaseLSTMModel(input_size=full_dataset_mv.features.shape[1], hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)

train_model(uni_model, train_loader_uni, epochs, device, lr)
train_model(mv_model, train_loader_mv, epochs, device, lr)

uni_metrics = evaluate_model(uni_model, test_loader_uni, full_dataset.scaler_y)
mv_metrics = evaluate_model(mv_model, test_loader_mv, full_dataset_mv.scaler_y)

comparison_df = pd.DataFrame([
    {"mode": "univariat", **uni_metrics, "commodity": series_name},
    {"mode": "multivariat", **mv_metrics, "commodity": series_name},
])
display(comparison_df)


/home/rna_13/anaconda3/envs/rapids-24.10/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/home/rna_13/anaconda3/envs/rapids-24.10/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


,mode,mae,rmse,mape,smape,directional_accuracy,commodity
0,univariat,94.435084,200.487189,0.851274,0.854015,89.677419,Beras Medium
1,multivariat,101.253440,197.313502,0.942984,0.944360,86.451613,Beras Medium


## Insight dari Notebook Modeling

**Ringkasan Temuan Utama**:

- Model multivariat yang menambahkan fitur produksi kuartal menunjukkan potensi peningkatan informasi dibandingkan model univariat yang hanya menggunakan harga historis. Perbedaan performa perlu divalidasi lebih luas menggunakan pelatihan yang lebih lama dan cross-validation.

- Penggabungan data produksi dilakukan dengan kunci `Komoditi` dan `quarter_start`; pastikan setiap pengamatan harga disejajarkan ke kuartal produksi yang benar agar fitur eksogen relevan.

- `PriceSequenceDataset` dibuat fleksibel untuk menerima input 1-dimensi atau multi-dimensi sehingga pipeline dapat digunakan ulang untuk kedua skenario tanpa perubahan arsitektur dataset.

- Catatan eksperimen: percobaan singkat di notebook menggunakan `epochs` kecil (demo). Untuk penilaian nyata, jalankan pelatihan penuh (mis. epochs=60+), simpan kurva training/validation, dan gunakan metric-driven early stopping.

- Rekomendasi praktis:
  1. Lakukan perbandingan terstandardisasi univariat vs multivariat per komoditas dan laporkan MAE/RMSE serta sMAPE.
  2. Tambahkan fitur lag, seasonality (sin/cos) dan flag libur untuk meningkatkan sinyal temporal.
  3. Jalankan Optuna per komoditas untuk menemukan konfigurasi LSTM terbaik dan simpan studi serta checkpoint.
  4. Validasi akhir pada horizon produksi nyata sebelum mempertimbangkan deployment.
